# Pre-train AMRBART with MBart-50 (Vietnamese)

This notebook runs the 6-task AMR pre-training on Google Colab with a single GPU.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repository

In [ ]:
!rm -rf /content/AMRBART
!git clone -b feat/mbart-50 https://github.com/Phucgiacat/AMRBART.git /content/AMRBART
!ls /content/AMRBART/pre-train/

## 3. Install dependencies

In [ ]:
# Add deadsnakes PPA and install Python 3.8
!apt-get update -qq 2>/dev/null
!apt-get install -qq -y software-properties-common 2>/dev/null
!add-apt-repository -y ppa:deadsnakes/ppa 2>/dev/null
!apt-get update -qq 2>/dev/null
!apt-get install -qq -y python3.8 python3.8-venv python3.8-dev python3.8-distutils

# Create a Python 3.8 venv
!python3.8 -m venv /content/py38env
!/content/py38env/bin/python --version

# Upgrade pip
!/content/py38env/bin/pip install --upgrade pip

# Install PyTorch 1.8.1 + CUDA 11.1 (Colab GPU driver is backward-compatible)
!/content/py38env/bin/pip install torch==1.8.1+cu111 torchvision==0.9.1+cu111 torchaudio==0.8.1 \
  -f https://download.pytorch.org/whl/torch_stable.html

# Install all other dependencies
!/content/py38env/bin/pip install -r /content/AMRBART/pre-train/requirements.txt

## 4. Download & prepare data

In [ ]:
!pip install -q gdown
!gdown --folder https://drive.google.com/drive/folders/10gwxpxAfha9zd1q6nakBBMohSBXGmmbC?usp=sharing -O /content/data

In [ ]:
!mkdir -p /content/AMRBART/pre-train/data/ViAMR
!cp /content/data/*.jsonl /content/AMRBART/pre-train/data/ViAMR/
!cp /content/AMRBART/pre-train/data/ViAMR/dev.jsonl /content/AMRBART/pre-train/data/ViAMR/val.jsonl
!ls -la /content/AMRBART/pre-train/data/ViAMR/

## 5. Download MBart-50 model

In [ ]:
!pip install -q huggingface_hub
!huggingface-cli download facebook/mbart-large-50 --local-dir /content/mbart-large-50

## 6. Verify setup

In [ ]:
!/content/py38env/bin/python -c "
import torch, transformers, datasets
print(f'Python:       {__import__(\"sys\").version}')
print(f'torch:        {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'datasets:     {datasets.__version__}')
print(f'CUDA:         {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"N/A\"})')
"

## 7. Run pre-training

In [ ]:
import os

os.environ['VENV'] = '/content/py38env'
os.environ['MODEL'] = '/content/mbart-large-50'

print(f"VENV={os.environ['VENV']}")
print(f"python={os.environ['VENV']}/bin/python")

!cd /content/AMRBART/pre-train && bash run-posttrain-mbart50-vietnamese-6task-large-unified.sh